In [33]:
# Install wandb for experiment tracking, transformers for AST, timm for ViT.
# Run this cell FIRST — Kaggle kernels may not have all of these by default.

import subprocess
subprocess.run(["pip", "install", "wandb", "transformers", "timm", "scikit-image", "-q"])

CompletedProcess(args=['pip', 'install', 'wandb', 'transformers', 'timm', 'scikit-image', '-q'], returncode=0)

In [34]:
!pip install transformers librosa soundfile wandb -q

In [56]:
class Config:
    # ── Paths ──
    BASE_DIR     = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup"
    GENRES_STEMS = f"{BASE_DIR}/genres_stems"
    ESC50_DIR    = f"{BASE_DIR}/ESC-50-master"
    MASHUPS_DIR  = f"{BASE_DIR}/mashups"
    TEST_CSV     = f"{BASE_DIR}/test.csv"
    OUTPUT_DIR   = "/kaggle/working"

    # ── Audio — MUST match AST pretrain spec ──
    SAMPLE_RATE  = 16000   # AST was pretrained at 16kHz — critical
    DURATION     = 10.0
    # AST feature extractor produces (1024, 128) mel patches automatically
    # Do NOT manually build mels for AST — use the feature extractor

    # ── Training ──
    SEED         = 42
         
    
   

   

    # ── WandB ──
    WANDB_PROJECT = "23f3004501-t12026"
    WANDB_ENTITY  = "23f3004501-indian-institute-of-technology-madras"
    WANDB_API_KEY = "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"
    GENRES = ["blues","classical","country","disco","hiphop",
              "jazz","metal","pop","reggae","rock"]
    NUM_CLASSES = 10

cfg = Config()

random.seed(cfg.SEED)
np.random.seed(cfg.SEED)
torch.manual_seed(cfg.SEED)
torch.cuda.manual_seed_all(cfg.SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE} ✓")
print(f"Sample rate for AST: {cfg.SAMPLE_RATE} Hz (must be 16000) ✓")

Device: cuda ✓
Sample rate for AST: 16000 Hz (must be 16000) ✓


In [57]:
import os, glob, random, numpy as np, pandas as pd
import librosa, torch, soundfile as sf
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    ASTFeatureExtractor, ASTForAudioClassification,
    Wav2Vec2FeatureExtractor, Wav2Vec2ForSequenceClassification,
    get_cosine_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.metrics import f1_score, classification_report, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import wandb

# ── Labels ────────────────────────────────────────────────────────────────────
GENRES   = ['blues','classical','country','disco','hiphop',
             'jazz','metal','pop','reggae','rock']
GENRE2ID = {g:i for i,g in enumerate(GENRES)}
ID2GENRE = {i:g for g,i in GENRE2ID.items()}

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT       = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_DIR  = f'{ROOT}/genres_stems'
ESC50_DIR  = f'{ROOT}/ESC-50-master/audio'
TEST_CSV   = f'{ROOT}/test.csv'

# ── Global Config ─────────────────────────────────────────────────────────────
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SR       = 16000
DURATION = 10
SEED     = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print('Device:', DEVICE)

Device: cuda


In [1]:
# ── Build stem index ──────────────────────────────────────────────────────────
stem_index = {genre: [] for genre in GENRES}
for genre in GENRES:
    genre_dir = os.path.join(STEMS_DIR, genre)
    for d in sorted(glob.glob(os.path.join(genre_dir, '*'))):
        if os.path.isdir(d):
            stem_index[genre].append(d)

for g, songs in stem_index.items():
    print(f'{g}: {len(songs)} songs')

# ── ESC-50 noise files ────────────────────────────────────────────────────────
esc50_files = glob.glob(os.path.join(ESC50_DIR, '*.wav'))
print(f'\nESC-50 noise files: {len(esc50_files)}')

NameError: name 'GENRES' is not defined

In [60]:
# ── Audio Utility Functions ───────────────────────────────────────────────────

def load_stem(song_dir, stem_name, sr=SR, duration=DURATION):
    """Load a single stem wav file."""
    path = os.path.join(song_dir, stem_name)
    if not os.path.exists(path):
        return np.zeros(sr * duration, dtype=np.float32)
    y, _ = librosa.load(path, sr=sr, duration=duration, mono=True)
    return y.astype(np.float32)

def pad_or_trim(audio, target_len):
    """Pad or trim audio to fixed length."""
    if len(audio) < target_len:
        return np.pad(audio, (0, target_len - len(audio)))
    return audio[:target_len]

def cross_song_mix(genre, sr=SR, duration=DURATION):
    """
    Core augmentation: pick 4 different songs from same genre,
    take one stem from each — replicates test mashup distribution.
    """
    songs = stem_index[genre]
    stem_names = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
    target_len = sr * duration
    chosen = random.choices(songs, k=4)
    mixed = np.zeros(target_len, dtype=np.float32)
    for song_dir, stem_name in zip(chosen, stem_names):
        stem = pad_or_trim(load_stem(song_dir, stem_name, sr, duration), target_len)
        mixed += stem
    mixed = mixed / (np.abs(mixed).max() + 1e-8)
    return mixed

def same_song_mix(song_dir, sr=SR, duration=DURATION):
    """Mix all 4 stems from the same song."""
    target_len = sr * duration
    mixed = np.zeros(target_len, dtype=np.float32)
    for stem_name in ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']:
        stem = pad_or_trim(load_stem(song_dir, stem_name, sr, duration), target_len)
        mixed += stem
    mixed = mixed / (np.abs(mixed).max() + 1e-8)
    return mixed

def add_esc50_noise(audio, esc50_files, sr=SR):
    """Add random ESC-50 noise at random SNR between 10-30 dB."""
    if not esc50_files:
        return audio
    noise_path = random.choice(esc50_files)
    try:
        noise, _ = librosa.load(noise_path, sr=sr, mono=True)
        noise = pad_or_trim(noise, len(audio))
        snr = random.uniform(10, 30)
        signal_power = np.mean(audio ** 2) + 1e-8
        noise_power  = np.mean(noise ** 2) + 1e-8
        scale = np.sqrt(signal_power / (noise_power * (10 ** (snr / 10))))
        noisy = audio + scale * noise
        return noisy / (np.abs(noisy).max() + 1e-8)
    except:
        return audio

print('Audio utility functions ready.')

Audio utility functions ready.


In [61]:
# ── Train/Val Split ───────────────────────────────────────────────────────────
records = []
for genre in GENRES:
    for song_dir in stem_index[genre]:
        records.append({'path': song_dir, 'label': GENRE2ID[genre], 'genre': genre})

random.shuffle(records)
split = int(0.85 * len(records))
train_records = records[:split]
val_records   = records[split:]
print(f'Total: {len(records)} | Train: {len(train_records)} | Val: {len(val_records)}')

Total: 1000 | Train: 850 | Val: 150


---
## MODEL 3: Classical ML — Logistic Regression (No GPU needed)
---

In [62]:
def extract_mfcc_features(song_dir, genre, augment=False, sr=SR, duration=DURATION):
    """Extract MFCC features from audio for classical ML."""
    if augment and random.random() < 0.8:
        audio = cross_song_mix(genre)
    else:
        audio = same_song_mix(song_dir)
    
    if augment and random.random() < 0.7:
        audio = add_esc50_noise(audio, esc50_files)
    
    audio = pad_or_trim(audio, sr * duration)
    
    # Extract MFCC features
    mfcc        = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
    chroma      = librosa.feature.chroma_stft(y=audio, sr=sr)
    spectral    = librosa.feature.spectral_contrast(y=audio, sr=sr)
    
    # Statistical aggregation
    features = np.concatenate([
        mfcc.mean(axis=1), mfcc.std(axis=1),
        chroma.mean(axis=1), chroma.std(axis=1),
        spectral.mean(axis=1), spectral.std(axis=1)
    ])
    return features

print('Extracting MFCC features for Classical ML...')
print('This takes ~5-10 mins...')

X_train, y_train = [], []
for rec in tqdm(train_records, desc='Train features'):
    feat = extract_mfcc_features(rec['path'], rec['genre'], augment=True)
    X_train.append(feat)
    y_train.append(rec['label'])

X_val, y_val = [], []
for rec in tqdm(val_records, desc='Val features'):
    feat = extract_mfcc_features(rec['path'], rec['genre'], augment=False)
    X_val.append(feat)
    y_val.append(rec['label'])

X_train = np.array(X_train)
X_val   = np.array(X_val)
y_train = np.array(y_train)
y_val   = np.array(y_val)

print(f'Feature shape: {X_train.shape}')

Extracting MFCC features for Classical ML...
This takes ~5-10 mins...


Val features: 100%|██████████| 150/150 [00:31<00:00,  4.70it/s]

Feature shape: (850, 118)


In [63]:
# ── Train Logistic Regression ─────────────────────────────────────────────────
wandb.init(project=WANDB_PROJECT, name="classical-ml-logistic-regression", reinit=True)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)

lr_model = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED)
lr_model.fit(X_train_scaled, y_train)

lr_preds = lr_model.predict(X_val_scaled)
lr_f1    = f1_score(y_val, lr_preds, average='macro')
lr_acc   = accuracy_score(y_val, lr_preds)

wandb.log({"val_f1": lr_f1, "val_accuracy": lr_acc, "model": "Logistic Regression"})
wandb.finish()

print(f'Logistic Regression | Val F1: {lr_f1:.4f} | Accuracy: {lr_acc:.4f}')
print(classification_report(y_val, lr_preds, target_names=GENRES))

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


val_accuracy,▁
val_f1,▁
model,Logistic Regression
val_accuracy,0.62
val_f1,0.60096


Logistic Regression | Val F1: 0.6010 | Accuracy: 0.6200
              precision    recall  f1-score   support

       blues       0.53      0.47      0.50        17
   classical       0.92      1.00      0.96        12
     country       0.70      0.64      0.67        22
       disco       0.55      0.40      0.46        15
      hiphop       0.80      0.44      0.57         9
        jazz       0.59      0.91      0.71        11
       metal       0.74      0.94      0.83        18
         pop       0.68      0.72      0.70        18
      reggae       0.50      0.15      0.24        13
        rock       0.30      0.47      0.37        15

    accuracy                           0.62       150
   macro avg       0.63      0.61      0.60       150
weighted avg       0.63      0.62      0.61       150



---
## MODEL 2: CNN FROM SCRATCH
---

In [64]:
CNN_EPOCHS = 15
CNN_LR     = 1e-3
CNN_BATCH  = 32
N_MELS     = 128

class MelDataset(Dataset):
    """Dataset that converts audio to Mel Spectrograms for CNN."""
    def __init__(self, records, augment=False):
        self.records = records
        self.augment = augment

    def __len__(self):
        return len(self.records) * (3 if self.augment else 1)

    def __getitem__(self, idx):
        rec = self.records[idx % len(self.records)]

        if self.augment and random.random() < 0.8:
            audio = cross_song_mix(rec['genre'])
        else:
            audio = same_song_mix(rec['path'])

        if self.augment and random.random() < 0.7:
            audio = add_esc50_noise(audio, esc50_files)

        audio = pad_or_trim(audio.astype(np.float32), SR * DURATION)

        # Convert to mel spectrogram
        mel    = librosa.feature.melspectrogram(y=audio, sr=SR, n_mels=N_MELS, fmax=8000)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-8)
        return torch.FloatTensor(mel_db).unsqueeze(0), rec['label']

cnn_train_ds = MelDataset(train_records, augment=True)
cnn_val_ds   = MelDataset(val_records,   augment=False)
cnn_train_loader = DataLoader(cnn_train_ds, batch_size=CNN_BATCH, shuffle=True,  num_workers=2)
cnn_val_loader   = DataLoader(cnn_val_ds,   batch_size=CNN_BATCH, shuffle=False, num_workers=2)
print('CNN Dataloaders ready.')

CNN Dataloaders ready.


In [65]:
class MusicCNN(nn.Module):
    """CNN built from scratch for music genre classification."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.25)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.25)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.25)
        )
        self.conv4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.25)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(256, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        return self.classifier(x)

cnn_model = MusicCNN(num_classes=10).to(DEVICE)
total_params = sum(p.numel() for p in cnn_model.parameters())
print(f'CNN Parameters: {total_params:,}')

CNN Parameters: 525,514


In [66]:
wandb.init(project=WANDB_PROJECT, name="cnn-from-scratch", reinit=True)

cnn_optimizer = torch.optim.Adam(cnn_model.parameters(), lr=CNN_LR)
cnn_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(cnn_optimizer, patience=3, factor=0.5)
cnn_criterion = nn.CrossEntropyLoss()
best_cnn_f1   = 0.0

for epoch in range(CNN_EPOCHS):
    cnn_model.train()
    total_loss = 0.0
    for inputs, labels in tqdm(cnn_train_loader, desc=f'CNN Epoch {epoch+1}/{CNN_EPOCHS}'):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        cnn_optimizer.zero_grad()
        loss = cnn_criterion(cnn_model(inputs), labels)
        loss.backward()
        cnn_optimizer.step()
        total_loss += loss.item()

    cnn_model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in cnn_val_loader:
            preds = cnn_model(inputs.to(DEVICE)).argmax(-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    f1       = f1_score(all_labels, all_preds, average='macro')
    acc      = accuracy_score(all_labels, all_preds)
    avg_loss = total_loss / len(cnn_train_loader)
    cnn_scheduler.step(avg_loss)

    wandb.log({"cnn_loss": avg_loss, "cnn_f1": f1, "cnn_accuracy": acc, "epoch": epoch+1})
    print(f'CNN Epoch {epoch+1:02d} | Loss: {avg_loss:.4f} | F1: {f1:.4f} | Acc: {acc:.4f}')

    if f1 > best_cnn_f1:
        best_cnn_f1 = f1
        torch.save(cnn_model.state_dict(), '/kaggle/working/best_cnn.pt')
        print(f'  >>> Best CNN saved (F1={f1:.4f})')

wandb.finish()
print(f'CNN done! Best F1: {best_cnn_f1:.4f}')

CNN Epoch 1/15: 100%|██████████| 80/80 [05:00<00:00,  3.76s/it]


CNN Epoch 01 | Loss: 2.1622 | F1: 0.2480 | Acc: 0.3067
  >>> Best CNN saved (F1=0.2480)


CNN Epoch 2/15: 100%|██████████| 80/80 [04:50<00:00,  3.63s/it]


CNN Epoch 02 | Loss: 1.9773 | F1: 0.2755 | Acc: 0.3067
  >>> Best CNN saved (F1=0.2755)


CNN Epoch 3/15: 100%|██████████| 80/80 [04:47<00:00,  3.59s/it]


...

CNN Epoch 09 | Loss: 1.6134 | F1: 0.4504 | Acc: 0.4800
  >>> Best CNN saved (F1=0.4504)


CNN Epoch 10/15: 100%|██████████| 80/80 [04:41<00:00,  3.52s/it]


CNN Epoch 10 | Loss: 1.5670 | F1: 0.3491 | Acc: 0.4000


CNN Epoch 11/15:   5%|▌         | 4/80 [00:14<04:41,  3.70s/it]


KeyboardInterrupt: 

---
## MODEL 1: AST — Pretrained Fine-tuning
---

In [67]:
AST_EPOCHS = 15
AST_BATCH  = 8
AST_LR     = 2e-5

ast_feature_extractor = ASTFeatureExtractor.from_pretrained(
    'MIT/ast-finetuned-audioset-10-10-0.4593'
)

class ASTDataset(Dataset):
    """Dataset for AST model using raw waveform input."""
    def __init__(self, records, augment=False):
        self.records   = records
        self.augment   = augment
        self.target_len = SR * DURATION

    def __len__(self):
        return len(self.records) * (3 if self.augment else 1)

    def __getitem__(self, idx):
        rec = self.records[idx % len(self.records)]

        if self.augment and random.random() < 0.8:
            audio = cross_song_mix(rec['genre'])
        else:
            audio = same_song_mix(rec['path'])

        if self.augment and random.random() < 0.7:
            audio = add_esc50_noise(audio, esc50_files)

        if self.augment:
            audio = audio * random.uniform(0.8, 1.2)
            audio = audio + np.random.randn(len(audio)).astype(np.float32) * 0.003

        audio = pad_or_trim(audio.astype(np.float32), self.target_len)
        inputs = ast_feature_extractor(audio, sampling_rate=SR, return_tensors='pt')
        return inputs['input_values'].squeeze(0), rec['label']

ast_train_ds = ASTDataset(train_records, augment=True)
ast_val_ds   = ASTDataset(val_records,   augment=False)
ast_train_loader = DataLoader(ast_train_ds, batch_size=AST_BATCH, shuffle=True,  num_workers=2, pin_memory=True)
ast_val_loader   = DataLoader(ast_val_ds,   batch_size=AST_BATCH, shuffle=False, num_workers=2, pin_memory=True)
print('AST Dataloaders ready.')

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

AST Dataloaders ready.


In [68]:
ast_model = ASTForAudioClassification.from_pretrained(
    'MIT/ast-finetuned-audioset-10-10-0.4593',
    num_labels=10,
    ignore_mismatched_sizes=True
).to(DEVICE)

ast_optimizer = AdamW(ast_model.parameters(), lr=AST_LR, weight_decay=0.01)
total_steps   = len(ast_train_loader) * AST_EPOCHS
warmup_steps  = total_steps // 10
ast_scheduler = get_cosine_schedule_with_warmup(
    ast_optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
ast_criterion = nn.CrossEntropyLoss()
print('AST model loaded!')

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


AST model loaded!


In [69]:
wandb.init(project=WANDB_PROJECT, name="ast-pretrained-finetuned", reinit=True)

best_ast_f1 = 0.0

for epoch in range(AST_EPOCHS):
    ast_model.train()
    total_loss = 0.0
    for inputs, labels in tqdm(ast_train_loader, desc=f'AST Epoch {epoch+1}/{AST_EPOCHS}'):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        ast_optimizer.zero_grad()
        loss = ast_criterion(ast_model(input_values=inputs).logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(ast_model.parameters(), 1.0)
        ast_optimizer.step()
        ast_scheduler.step()
        total_loss += loss.item()

    ast_model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in tqdm(ast_val_loader, desc=f'AST Val {epoch+1}'):
            preds = ast_model(input_values=inputs.to(DEVICE)).logits.argmax(-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    f1       = f1_score(all_labels, all_preds, average='macro')
    acc      = accuracy_score(all_labels, all_preds)
    avg_loss = total_loss / len(ast_train_loader)

    wandb.log({"ast_loss": avg_loss, "ast_f1": f1, "ast_accuracy": acc, "epoch": epoch+1})
    print(f'AST Epoch {epoch+1:02d} | Loss: {avg_loss:.4f} | F1: {f1:.4f} | Acc: {acc:.4f}')

    if f1 > best_ast_f1:
        best_ast_f1 = f1
        torch.save(ast_model.state_dict(), '/kaggle/working/best_ast.pt')
        print(f'  >>> Best AST saved (F1={f1:.4f})')

wandb.finish()
print(f'AST done! Best F1: {best_ast_f1:.4f}')

cnn_accuracy,▃▃▁▃▇█▇██▆
cnn_f1,▂▃▁▃▇█▆██▅
cnn_loss,█▆▅▄▃▃▃▂▂▁
epoch,▁▂▃▃▄▅▆▆▇█
cnn_accuracy,0.4
cnn_f1,0.3491
cnn_loss,1.56695
epoch,10


AST Val 1: 100%|██████████| 19/19 [00:16<00:00,  1.18it/s]


AST Epoch 01 | Loss: 1.1757 | F1: 0.7743 | Acc: 0.7733
  >>> Best AST saved (F1=0.7743)


AST Val 2: 100%|██████████| 19/19 [00:16<00:00,  1.18it/s]


AST Epoch 02 | Loss: 0.3566 | F1: 0.8500 | Acc: 0.8533
  >>> Best AST saved (F1=0.8500)


...

AST Epoch 14 | Loss: 0.0091 | F1: 0.9673 | Acc: 0.9667


AST Val 15: 100%|██████████| 19/19 [00:15<00:00,  1.20it/s]

AST Epoch 15 | Loss: 0.0032 | F1: 0.9673 | Acc: 0.9667


ast_accuracy,▁▄▅▅▄▆▇▇█▇▇████
ast_f1,▁▄▅▅▄▆▇▇▇▇▇████
ast_loss,█▃▃▂▂▂▁▁▁▁▁▁▁▁▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
ast_accuracy,0.96667
ast_f1,0.96733
ast_loss,0.00316
epoch,15


AST done! Best F1: 0.9740


---
## MODEL 4: wav2vec2 — Second Pretrained Model for Ensemble
---

In [ ]:
W2V_EPOCHS = 7
W2V_BATCH  = 8
W2V_LR     = 2e-5

w2v_feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
    'facebook/wav2vec2-base'
)

class W2VDataset(Dataset):
    """Dataset for wav2vec2 model."""
    def __init__(self, records, augment=False):
        self.records    = records
        self.augment    = augment
        self.target_len = SR * DURATION

    def __len__(self):
        return len(self.records) * (3 if self.augment else 1)

    def __getitem__(self, idx):
        rec = self.records[idx % len(self.records)]

        if self.augment and random.random() < 0.8:
            audio = cross_song_mix(rec['genre'])
        else:
            audio = same_song_mix(rec['path'])

        if self.augment and random.random() < 0.7:
            audio = add_esc50_noise(audio, esc50_files)

        audio = pad_or_trim(audio.astype(np.float32), self.target_len)
        inputs = w2v_feature_extractor(
            audio, sampling_rate=SR, return_tensors='pt', padding=True
        )
        return inputs['input_values'].squeeze(0), rec['label']

w2v_train_ds = W2VDataset(train_records, augment=True)
w2v_val_ds   = W2VDataset(val_records,   augment=False)
w2v_train_loader = DataLoader(w2v_train_ds, batch_size=W2V_BATCH, shuffle=True,  num_workers=2, pin_memory=True)
w2v_val_loader   = DataLoader(w2v_val_ds,   batch_size=W2V_BATCH, shuffle=False, num_workers=2, pin_memory=True)
print('wav2vec2 Dataloaders ready.')

In [71]:
w2v_model = Wav2Vec2ForSequenceClassification.from_pretrained(
    'facebook/wav2vec2-base',
    num_labels=10,
    ignore_mismatched_sizes=True
).to(DEVICE)

w2v_optimizer = AdamW(w2v_model.parameters(), lr=W2V_LR, weight_decay=0.01)
w2v_total_steps  = len(w2v_train_loader) * W2V_EPOCHS
w2v_warmup_steps = w2v_total_steps // 10
w2v_scheduler = get_cosine_schedule_with_warmup(
    w2v_optimizer, num_warmup_steps=w2v_warmup_steps, num_training_steps=w2v_total_steps
)
w2v_criterion = nn.CrossEntropyLoss()
print('wav2vec2 model loaded!')

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

wav2vec2 model loaded!


In [72]:
wandb.init(project=WANDB_PROJECT, name="wav2vec2-finetuned", reinit=True)

best_w2v_f1 = 0.0

for epoch in range(W2V_EPOCHS):
    w2v_model.train()
    total_loss = 0.0
    for inputs, labels in tqdm(w2v_train_loader, desc=f'W2V Epoch {epoch+1}/{W2V_EPOCHS}'):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        w2v_optimizer.zero_grad()
        loss = w2v_criterion(w2v_model(input_values=inputs).logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(w2v_model.parameters(), 1.0)
        w2v_optimizer.step()
        w2v_scheduler.step()
        total_loss += loss.item()

    w2v_model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in w2v_val_loader:
            preds = w2v_model(input_values=inputs.to(DEVICE)).logits.argmax(-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    f1       = f1_score(all_labels, all_preds, average='macro')
    acc      = accuracy_score(all_labels, all_preds)
    avg_loss = total_loss / len(w2v_train_loader)

    wandb.log({"w2v_loss": avg_loss, "w2v_f1": f1, "w2v_accuracy": acc, "epoch": epoch+1})
    print(f'W2V Epoch {epoch+1:02d} | Loss: {avg_loss:.4f} | F1: {f1:.4f} | Acc: {acc:.4f}')

    if f1 > best_w2v_f1:
        best_w2v_f1 = f1
        torch.save(w2v_model.state_dict(), '/kaggle/working/best_w2v.pt')
        print(f'  >>> Best W2V saved (F1={f1:.4f})')

wandb.finish()
print(f'wav2vec2 done! Best F1: {best_w2v_f1:.4f}')

W2V Epoch 1/10: 100%|██████████| 319/319 [11:43<00:00,  2.20s/it]


W2V Epoch 01 | Loss: 2.0836 | F1: 0.3016 | Acc: 0.4133
  >>> Best W2V saved (F1=0.3016)


W2V Epoch 2/10: 100%|██████████| 319/319 [11:42<00:00,  2.20s/it]


W2V Epoch 02 | Loss: 1.5272 | F1: 0.5664 | Acc: 0.6000
  >>> Best W2V saved (F1=0.5664)


W2V Epoch 3/10: 100%|██████████| 319/319 [11:42<00:00,  2.20s/it]


...

W2V Epoch 09 | Loss: 0.4403 | F1: 0.8195 | Acc: 0.8133
  >>> Best W2V saved (F1=0.8195)


W2V Epoch 10/10: 100%|██████████| 319/319 [11:41<00:00,  2.20s/it]


W2V Epoch 10 | Loss: 0.3951 | F1: 0.8207 | Acc: 0.8133
  >>> Best W2V saved (F1=0.8207)


epoch,▁▂▃▃▄▅▆▆▇█
w2v_accuracy,▁▄▅▇▇▇▇███
w2v_f1,▁▅▆▇▇▇▇███
w2v_loss,█▆▄▃▃▂▂▁▁▁
epoch,10
w2v_accuracy,0.81333
w2v_f1,0.82074
w2v_loss,0.39513


wav2vec2 done! Best F1: 0.8207


---
## ENSEMBLE INFERENCE — AST + wav2vec2
---

In [1]:
import os
print(os.path.exists('/kaggle/working/best_ast.pt'))
print(os.path.exists('/kaggle/working/best_w2v.pt'))

False
False


In [1]:
# Load best checkpoints
ast_model.load_state_dict(torch.load('/kaggle/working/best_ast.pt'))
ast_model.eval()

w2v_model.load_state_dict(torch.load('/kaggle/working/best_w2v.pt'))
w2v_model.eval()

print('Both models loaded for ensemble inference!')

NameError: name 'ast_model' is not defined

In [4]:
def ensemble_infer(audio_path, n_tta=10):
    """
    Ensemble inference: AST + wav2vec2 logits averaged.
    TTA: run 5 times with slight variations.
    """
    try:
        audio, _ = librosa.load(audio_path, sr=SR, duration=DURATION, mono=True)
        audio    = pad_or_trim(audio.astype(np.float32), SR * DURATION)
    except Exception as e:
        print(f'Load error: {e}')
        return 'rock'

    ast_logits_sum = None
    w2v_logits_sum = None

    for i in range(n_tta):
        if i == 0:
            aug = audio.copy()
        else:
            aug = audio + np.random.randn(len(audio)).astype(np.float32) * 0.003
            aug = aug * random.uniform(0.9, 1.1)
            aug = aug / (np.abs(aug).max() + 1e-8)

        # AST inference
        ast_inputs = ast_feature_extractor(aug, sampling_rate=SR, return_tensors='pt')
        with torch.no_grad():
            ast_logits = ast_model(input_values=ast_inputs['input_values'].to(DEVICE)).logits
        ast_logits_sum = ast_logits if ast_logits_sum is None else ast_logits_sum + ast_logits

        # wav2vec2 inference
        w2v_inputs = w2v_feature_extractor(aug, sampling_rate=SR, return_tensors='pt', padding=True)
        with torch.no_grad():
            w2v_logits = w2v_model(input_values=w2v_inputs['input_values'].to(DEVICE)).logits
        w2v_logits_sum = w2v_logits if w2v_logits_sum is None else w2v_logits_sum + w2v_logits

    # Ensemble: average AST and wav2vec2 logits
    ensemble_logits = ast_logits_sum + w2v_logits_sum
    pred_id = ensemble_logits.argmax(-1).item()
    return ID2GENRE[pred_id]

print('Ensemble inference function ready.')

Ensemble inference function ready.


In [ ]:
test_df = pd.read_csv(TEST_CSV)
print(f'Test samples: {len(test_df)}')

predictions = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Ensemble Inference'):
    audio_path = os.path.join(ROOT, row['filename'])
    pred = ensemble_infer(audio_path, n_tta=5)
    predictions.append(pred)

print('Done! Distribution:')
print(pd.Series(predictions).value_counts())

In [ ]:
sub = pd.DataFrame({'id': test_df['id'], 'genre': predictions})
sub.to_csv('/kaggle/working/submission.csv', index=False)
print('submission.csv saved!')
print(sub.head(10))

In [ ]:
from IPython.display import FileLink
display(FileLink('/kaggle/working/submission.csv'))